# CBIS-DDSM Data Preparation (standalone)

> **Prefer the single pipeline notebook:** `cbis_ddsm_deepproblog_pipeline.ipynb` (data prep + DeepProbLog in one Run All).

This notebook is **metadata preparation only**. It does **not** train DeepProbLog.

- Respects the official CBIS-DDSM train/test partition
- Validation is split from official **train** patients only (`GroupShuffleSplit` by `patient_id`)
- Set `SMOKE_TEST = True` to run on a tiny patient subset for quick checks

In [ ]:
# Optional: install only if imports fail on your environment
# %pip install -q kagglehub[pandas-datasets] scikit-learn pandas

In [2]:
import json
import os
import random
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

pandas: 2.3.3
numpy: 2.0.2


In [ ]:
# --- Configuration ---
SMOKE_TEST = True  # flip to False for full run
SMOKE_N_PATIENTS = 20  # max patients sampled per official split when smoke testing
VAL_SIZE = 0.15  # validation fraction from OFFICIAL TRAIN patients
RANDOM_SEED = 42

DATASET_HANDLE = "awsaf49/cbis-ddsm-breast-cancer-image-dataset"
KAGGLE_INPUT = "/kaggle/input/cbis-ddsm-breast-cancer-image-dataset"
LOCAL_DATA = Path("../data")

CSV_FILENAMES = {
    "mass_train": "mass_case_description_train_set.csv",
    "mass_test": "mass_case_description_test_set.csv",
    "calc_train": "calc_case_description_train_set.csv",
    "calc_test": "calc_case_description_test_set.csv",
}

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

if SMOKE_TEST:
    print("=" * 60)
    print("SMOKE_TEST=True — results are for pipeline checks only.")
    print("Do not use these counts for paper/experiment reporting.")
    print("=" * 60)

## Path discovery

Search Kaggle input, local `data/`, or `kagglehub` download for the four official CSV files.

In [3]:
def find_csv_paths(root: Path, filenames: dict[str, str]) -> dict[str, Path]:
    """Recursively locate CSV files under root (skip jpeg trees)."""
    root = Path(root)
    found: dict[str, Path] = {}
    remaining = set(filenames.values())

    for dirpath, dirnames, files in os.walk(root):
        dirnames[:] = [d for d in dirnames if d.lower() != "jpeg"]
        for fname in files:
            if fname in remaining:
                key = next(k for k, v in filenames.items() if v == fname)
                found[key] = Path(dirpath) / fname
                remaining.discard(fname)
        if not remaining:
            break

    missing = [k for k in filenames if k not in found]
    if missing:
        raise FileNotFoundError(
            f"Missing CSV keys {missing} under {root}. Found: {list(found.keys())}"
        )
    return found


def resolve_data_root() -> tuple[Path, str]:
    """Return (root_path, source_label)."""
    if Path(KAGGLE_INPUT).is_dir():
        return Path(KAGGLE_INPUT), "kaggle_input"
    if LOCAL_DATA.is_dir() and all((LOCAL_DATA / fn).exists() for fn in CSV_FILENAMES.values()):
        return LOCAL_DATA, "local_data"

    try:
        import kagglehub

        downloaded = Path(kagglehub.dataset_download(DATASET_HANDLE))
        return downloaded, "kagglehub_download"
    except ImportError as exc:
        raise FileNotFoundError(
            "No Kaggle input or local data found, and kagglehub is not installed."
        ) from exc


DATA_ROOT, DATA_SOURCE = resolve_data_root()
CSV_PATHS = find_csv_paths(DATA_ROOT, CSV_FILENAMES)

print(f"Data source: {DATA_SOURCE}")
print(f"Data root:   {DATA_ROOT}")
for key, path in CSV_PATHS.items():
    print(f"  {key}: {path}")

NameError: name 'KAGGLE_INPUT' is not defined

In [ ]:
def load_csv(path: Path) -> pd.DataFrame:
    """Load a CSV from disk or via kagglehub pandas adapter."""
    if path.is_file():
        return pd.read_csv(path)

    import kagglehub
    from kagglehub import KaggleDatasetAdapter

    rel = path.relative_to(DATA_ROOT).as_posix()
    return kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        DATASET_HANDLE,
        rel,
    )


raw = {key: load_csv(path) for key, path in CSV_PATHS.items()}
for key, df in raw.items():
    print(f"{key}: {len(df):>5} rows, {df['patient_id'].nunique():>4} patients")

## Harmonize columns and merge mass + calc

In [ ]:
STRING_COLUMNS = [
    "patient_id",
    "left or right breast",
    "image view",
    "abnormality type",
    "mass shape",
    "mass margins",
    "calc type",
    "calc distribution",
    "pathology",
    "image file path",
    "cropped image file path",
    "ROI mask file path",
]


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [c.strip() for c in out.columns]
    if "breast density" in out.columns:
        out = out.rename(columns={"breast density": "breast_density"})
    if "breast_density" not in out.columns and "breast density" not in out.columns:
        pass  # keep as-is if unexpected schema

    for col in STRING_COLUMNS:
        if col in out.columns:
            out[col] = out[col].astype(str).str.strip()
            out[col] = out[col].replace({"nan": np.nan, "None": np.nan, "": np.nan})
    return out


def add_record_metadata(
    df: pd.DataFrame, abnormality_kind: str, official_split: str
) -> pd.DataFrame:
    out = df.copy()
    out["abnormality_kind"] = abnormality_kind
    out["official_split"] = official_split
    return out


def merge_mass_calc(mass_df: pd.DataFrame, calc_df: pd.DataFrame) -> pd.DataFrame:
    return pd.concat([mass_df, calc_df], ignore_index=True, sort=False)


mass_train = add_record_metadata(
    normalize_columns(raw["mass_train"]), "mass", "train"
)
mass_test = add_record_metadata(
    normalize_columns(raw["mass_test"]), "mass", "test"
)
calc_train = add_record_metadata(
    normalize_columns(raw["calc_train"]), "calcification", "train"
)
calc_test = add_record_metadata(
    normalize_columns(raw["calc_test"]), "calcification", "test"
)

df_train_all = merge_mass_calc(mass_train, calc_train)
df_test_all = merge_mass_calc(mass_test, calc_test)

print("Merged official train:", len(df_train_all), "rows,", df_train_all["patient_id"].nunique(), "patients")
print("Merged official test: ", len(df_test_all), "rows,", df_test_all["patient_id"].nunique(), "patients")

## Smoke-test subsampling (by patient)

In [ ]:
def sample_patients(df: pd.DataFrame, n_patients: int, seed: int) -> pd.DataFrame:
    """Keep all rows for a random subset of patients."""
    patients = df["patient_id"].dropna().unique()
    if len(patients) <= n_patients:
        return df.copy()
    rng = random.Random(seed)
    chosen = set(rng.sample(list(patients), n_patients))
    return df[df["patient_id"].isin(chosen)].copy()


if SMOKE_TEST:
    df_train_all = sample_patients(df_train_all, SMOKE_N_PATIENTS, RANDOM_SEED)
    df_test_all = sample_patients(df_test_all, SMOKE_N_PATIENTS, RANDOM_SEED + 1)
    print(f"After smoke sampling (max {SMOKE_N_PATIENTS} patients per official split):")
    print("  train:", len(df_train_all), "rows,", df_train_all["patient_id"].nunique(), "patients")
    print("  test: ", len(df_test_all), "rows,", df_test_all["patient_id"].nunique(), "patients")

## Patient-level validation split (from official train only)

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=RANDOM_SEED)
train_idx, val_idx = next(gss.split(df_train_all, groups=df_train_all["patient_id"]))

df_train = df_train_all.iloc[train_idx].reset_index(drop=True)
df_val = df_train_all.iloc[val_idx].reset_index(drop=True)
df_test = df_test_all.reset_index(drop=True)

train_patients = set(df_train["patient_id"])
val_patients = set(df_val["patient_id"])
test_patients = set(df_test["patient_id"])

assert train_patients.isdisjoint(val_patients), "Train/val patient leakage"
assert train_patients.isdisjoint(test_patients), "Train/test patient leakage"
assert val_patients.isdisjoint(test_patients), "Val/test patient leakage"

for name, frame in [("train", df_train), ("val", df_val), ("test", df_test)]:
    n_pat = frame["patient_id"].nunique()
    multi = frame.groupby("patient_id").size()
    if (multi > 1).any():
        print(f"{name:5s}: max records/patient = {multi.max()}")
    print(f"{name:5s}: {len(frame):>5} rows | {n_pat:>4} patients")

print("\nLeakage checks passed: no patient appears in more than one split.")

In [ ]:
# Patients with both mass and calcification rows must stay in one split
for split_name, frame in [("train", df_train), ("val", df_val), ("test", df_test)]:
    kinds = frame.groupby("patient_id")["abnormality_kind"].nunique()
    dual = kinds[kinds > 1]
    if len(dual):
        print(f"{split_name}: {len(dual)} patients with both mass and calc records (OK — same split)")
    else:
        print(f"{split_name}: no dual-kind patients in this smoke/full subset")

## Summaries and optional path checks

In [ ]:
def split_summary(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "split": split_name,
            "rows": [len(df)],
            "patients": [df["patient_id"].nunique()],
            "mass_rows": [(df["abnormality_kind"] == "mass").sum()],
            "calc_rows": [(df["abnormality_kind"] == "calcification").sum()],
            "malignant_rows": [(df["pathology"] == "MALIGNANT").sum()],
            "benign_rows": [(df["pathology"] == "BENIGN").sum()],
        }
    )


summary = pd.concat(
    [
        split_summary(df_train, "train"),
        split_summary(df_val, "val"),
        split_summary(df_test, "test"),
    ],
    ignore_index=True,
)
display(summary)

print("\nPathology x abnormality_kind (train):")
display(pd.crosstab(df_train["pathology"], df_train["abnormality_kind"]))

In [ ]:
JPEG_ROOT = DATA_ROOT / "jpeg"
if not JPEG_ROOT.is_dir():
    # Some layouts nest jpeg differently
    candidates = list(DATA_ROOT.rglob("jpeg"))
    JPEG_ROOT = candidates[0] if candidates else JPEG_ROOT


def clean_path(value):
    if pd.isna(value):
        return None
    text = str(value).strip().strip('"').split("\n")[0].strip()
    return text or None


def resolve_image_path(row, prefer="cropped"):
    col_order = (
        ["cropped image file path", "image file path"]
        if prefer == "cropped"
        else ["image file path", "cropped image file path"]
    )
    rel = None
    for col in col_order:
        rel = clean_path(row.get(col))
        if rel:
            break
    if not rel:
        return None
    rel = rel.replace(".dcm", ".jpg")
    return JPEG_ROOT / rel


N_PATH_CHECKS = 10 if SMOKE_TEST else 0
if JPEG_ROOT.is_dir() and N_PATH_CHECKS:
    sample_rows = df_train.head(N_PATH_CHECKS)
    checks = []
    for _, row in sample_rows.iterrows():
        p = resolve_image_path(row)
        checks.append({"patient_id": row["patient_id"], "exists": p.is_file() if p else False, "path": str(p)})
    path_df = pd.DataFrame(checks)
    print(f"JPEG root: {JPEG_ROOT}")
    print(f"Existence check on first {N_PATH_CHECKS} train rows:")
    display(path_df)
else:
    print(f"Skipping path existence checks (jpeg root exists={JPEG_ROOT.is_dir()}).")

## Save processed artifacts

In [ ]:
if Path("/kaggle/working").is_dir():
    OUTPUT_DIR = Path("/kaggle/working/processed")
else:
    OUTPUT_DIR = Path("../data/processed")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

patient_splits = []
for split_name, frame in [("train", df_train), ("val", df_val), ("test", df_test)]:
    for pid in sorted(frame["patient_id"].dropna().unique()):
        patient_splits.append({"patient_id": pid, "split": split_name})
df_patient_splits = pd.DataFrame(patient_splits)

df_train.to_csv(OUTPUT_DIR / "cbis_train.csv", index=False)
df_val.to_csv(OUTPUT_DIR / "cbis_val.csv", index=False)
df_test.to_csv(OUTPUT_DIR / "cbis_test.csv", index=False)
df_patient_splits.to_csv(OUTPUT_DIR / "cbis_patient_splits.csv", index=False)

metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "smoke_test": SMOKE_TEST,
    "smoke_n_patients": SMOKE_N_PATIENTS if SMOKE_TEST else None,
    "val_size": VAL_SIZE,
    "random_seed": RANDOM_SEED,
    "data_source": DATA_SOURCE,
    "data_root": str(DATA_ROOT),
    "counts": {
        "train_rows": len(df_train),
        "val_rows": len(df_val),
        "test_rows": len(df_test),
        "train_patients": df_train["patient_id"].nunique(),
        "val_patients": df_val["patient_id"].nunique(),
        "test_patients": df_test["patient_id"].nunique(),
    },
}
with open(OUTPUT_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved processed files to: {OUTPUT_DIR.resolve()}")
for fname in ["cbis_train.csv", "cbis_val.csv", "cbis_test.csv", "cbis_patient_splits.csv", "metadata.json"]:
    print(" ", fname)

In [ ]:
print("Schema (train):")
display(df_train.dtypes.to_frame("dtype"))
print("\nFirst 5 train records:")
display(df_train.head())